# <span style="color: #00bfff">Unidad 5. Agentic AI</span>



# <span style="color: #00bfff">Agente con Patrón ReAct</span>

Esta notebook muestra la configuración y ejecución de un agente básico usando LangChain y Google Gemini. El agente sigue el patrón **ReAct**:
- **Reason (Razonamiento)**: El LLM analiza el problema y planea.
- **Act (Acción)**: Ejecuta herramientas (e.g., calculadora) para obtener datos.
- **Iteración**: Repite hasta resolver, incorporando observaciones.

**Objetivo del ejemplo**: Un agente que calcula el área de un círculo de radio 5, razonando paso a paso y usando una herramienta para el cálculo matemático.

## **Paso 1**: Instalación de dependencias y configuración de la API key.


In [ ]:
# Instalación de dependencias (Si no están instaladas ya)
!pip install langchain langchain-google-genai python-dotenv

In [4]:
# Importar librerías necesarias
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_react_agent, AgentExecutor
from langchain.tools import tool
from langchain.prompts import PromptTemplate

# Cargar variables de entorno (crea un archivo .env con GOOGLE_API_KEY=tu_clave)
load_dotenv()

# Configurar el LLM (Gemini)
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-preview-09-2025",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0  # Baja temperatura para razonamiento preciso
)

print("LLM configurado correctamente.") 

LLM configurado correctamente.


## **Paso 2**: Definir herramientas y el patrón ReAct

- **Herramienta**: Una función simple `calculate_area` que usa matemáticas para calcular el área (πr²). Esto simula una "acción" externa.
- **Patrón ReAct**: Usamos `create_react_agent` de LangChain, que genera prompts para razonar ("Thought") y actuar ("Action"). El agente itera: Thought → Action → Observation → Repeat.

El prompt guía al agente a usar la herramienta solo cuando sea necesario.

In [5]:
import math

# Definir una herramienta simple (acción: calcular área de círculo)
@tool
def calculate_area(radius: float) -> str:
    """
    Calcula el área de un círculo dado el radio.
    Args:
        radius (float): Radio del círculo.
    # Returns:
        str: Área calculada con explicación.
    """
    radius = float(radius)
    area = math.pi * radius ** 2
    return f"El área del círculo con radio {radius} es aproximadamente {area:.2f} unidades cuadradas."

# Lista de herramientas
tools = [calculate_area]

# Prompt para ReAct (guía el razonamiento y acción)
prompt = PromptTemplate.from_template("""
Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}""")

llm_with_stop = llm.bind(stop=["Observation"])

# Crear el agente ReAct
agent = create_react_agent(llm, tools, prompt)

# Executor para correr el agente
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, max_iterations=3, handle_parsing_errors=True)  # Límite de iteraciones para evitar loops

print("Herramienta y agente ReAct configurados.")

Herramienta y agente ReAct configurados.


## **Paso 3**: Ejecutar el agente

Ejecutamos el agente con una consulta: "Calcula el área de un círculo de radio 5 y explica por qué usas esa fórmula."

El agente:
1. **Razonamiento (Thought)**: Recuerda la fórmula del área (πr²).
2. **Acción (Action)**: Llama la herramienta con radio=5.
3. **Observación (Observation)**: Recibe el resultado.
4. **Finalización**: Sintetiza la explicación.

In [6]:
# Consulta del usuario
query = "Calcula el área de un círculo de radio 5 y explica por qué usas esa fórmula."

# Ejecutar el agente
result = agent_executor.invoke({"input": query})

# Mostrar resultado final
print("Respuesta final del agente:")
print(result['output'])



> Entering new AgentExecutor chain...
Thought: The user wants me to calculate the area of a circle with a radius of 5 and explain the formula used. I have a tool `calculate_area` that can perform the calculation. I will use the tool first, and then provide the explanation in the final answer.

Action: calculate_area
Action Input: 5El área del círculo con radio 5.0 es aproximadamente 78.54 unidades cuadradas.Thought: The user wants me to calculate the area of a circle with a radius of 5 and explain the formula used. I have a tool `calculate_area` that can perform the calculation. I will use the tool first, and then provide the explanation in the final answer.

Action: calculate_area
Action Input: 5El área del círculo con radio 5.0 es aproximadamente 78.54 unidades cuadradas.Thought: The user wants me to calculate the area of a circle with a radius of 5 and explain the formula used. I have a tool `calculate_area` that can perform the calculation. I will use the tool first, and then pro

## **Paso 4:** Análisis del funcionamiento

- **Funcionamiento**: El agente itera ReAct para razonar antes de actuar, usando el LLM (Gemini) para inteligencia y la herramienta para precisión matemática. Esto demuestra autonomía: no improvisa cálculos, sino que "piensa" y "actúa".
- **Configuración clave**: 
  - LLM: Maneja razonamiento natural.
  - Herramientas: Extienden capacidades (agrega más con `@tool`).
  - Prompt: Define el patrón ReAct para estructura.
- **Ventajas**: Reduce errores; adaptable (cambia consulta para tareas como "Busca clima" agregando una herramienta de API).
- **Limitaciones**: Depende de la API key; para producción, agrega manejo de errores y memoria persistente (usa LangChain's Memory).
